In [1]:
# ============================================================
# Verify GPU runtime
# ============================================================
# Why: Training on CPU takes 90+ minutes. T4 takes 5 minutes.
# If CUDA is False, stop and change the runtime before continuing.
# ============================================================

import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Memory: {props.total_memory / 1e9:.1f} GB")
else:
    print("No GPU. Runtime -> Change runtime type -> T4 GPU -> Save.")

PyTorch: 2.11.0+cpu
CUDA available: False
No GPU. Runtime -> Change runtime type -> T4 GPU -> Save.


In [3]:
# ============================================================
# net.py to Colab filesystem
# ============================================================
# Why: This file MUST be byte-identical to the local domain/model.py.
# Any drift between the two means baseline.pt will not load in serving.
#
# Why 4 classes: CIFAR-10 subset (airplane, automobile, bird, cat).
# Faster training, cleaner attack visualizations, no loss of
# demonstration value.
# ============================================================

%%writefile /content/net.py
"""
SmallCNN for 4-class CIFAR-10 subset.

Byte-identical to the local domain/model.py. Any drift breaks
state_dict loading in the serving pipeline.
"""
import torch
import torch.nn as nn


# CIFAR-10 subset: indices 0, 1, 2, 3
CLASSES = ["airplane", "automobile", "bird", "cat"]
NUM_CLASSES = len(CLASSES)

CIFAR_MEAN = (0.5, 0.5, 0.5)
CIFAR_STD = (0.5, 0.5, 0.5)


class SmallCNN(nn.Module):
    """
    3-layer CNN with dropout. ~600K params.

    Deliberately small: trains in under 10 minutes on a free T4,
    fits comfortably in 8GB RAM, and every layer is explainable
    line-by-line in a live code walkthrough.
    """

    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        # Convolutional feature extractor
        # Spatial dims: 32 -> 16 -> 8 -> 4
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        # Classifier head: 128 channels * 4 * 4 = 2048 features
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

Overwriting /content/net.py


In [4]:
import sys
sys.path.insert(0, '/content')
from net import SmallCNN, CLASSES

m = SmallCNN()
print("Classes:", CLASSES)
print("Parameters:", sum(p.numel() for p in m.parameters()))

Classes: ['airplane', 'automobile', 'bird', 'cat']
Parameters: 618820


In [6]:
# ============================================================
# train.py
# ============================================================
# Fix 1: Full seeding (torch, numpy, random) for reproducibility
# Fix 2: Validation split (10% of train) — no test-set leakage
# Fix 3: Export config.json with training metadata
# Fix 4: Export curve.json with per-epoch metrics
#
# Checkpoint selection is by validation accuracy, not test accuracy.
# Test set is evaluated once at the very end.
# ============================================================

%%writefile /content/train.py
"""
Train SmallCNN on the 4-class CIFAR-10 subset.

Fixes applied over the initial version:
  1. Seeds torch, numpy, and random for full reproducibility.
  2. Splits train into train/validation (90/10) to avoid test-set
     selection leakage.
  3. Saves checkpoints by validation accuracy, not test accuracy.
  4. Exports config.json and curve.json alongside baseline.pt.

Outputs to /content/:
  baseline.pt    - the trained weights
  config.json    - training metadata and normalization constants
  curve.json     - per-epoch loss and validation accuracy
"""
import os
import sys
import time
import json
import random
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

sys.path.insert(0, '/content')
from net import SmallCNN, CLASSES, CIFAR_MEAN, CIFAR_STD


# -----------------------------------------------------------
# Configuration
# -----------------------------------------------------------
TARGET_LABELS = [0, 1, 2, 3]      # airplane, automobile, bird, cat
OUT_DIR = '/content'
OUT_PATH = os.path.join(OUT_DIR, 'baseline.pt')
CONFIG_PATH = os.path.join(OUT_DIR, 'config.json')
CURVE_PATH = os.path.join(OUT_DIR, 'curve.json')

EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
VAL_FRACTION = 0.1
SEED = 42


# -----------------------------------------------------------
# Fix 1: Full seeding
# -----------------------------------------------------------
def set_seeds(seed: int = SEED) -> None:
    """Seed every source of randomness for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -----------------------------------------------------------
# Data loading with train/val/test split
# -----------------------------------------------------------
def get_loaders(batch_size: int = BATCH_SIZE, val_fraction: float = VAL_FRACTION):
    """
    Fix 2: separate validation from training.

    Returns (train_loader, val_loader, test_loader).
    The test loader is only touched at the end of training.
    """
    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    train_ds = datasets.CIFAR10(
        root='/content/data', train=True, download=True, transform=tfm)
    test_ds = datasets.CIFAR10(
        root='/content/data', train=False, download=True, transform=tfm)

    # Filter to the 4-class subset
    train_idx = [i for i, (_, y) in enumerate(train_ds) if y in TARGET_LABELS]
    test_idx = [i for i, (_, y) in enumerate(test_ds) if y in TARGET_LABELS]

    train_subset = Subset(train_ds, train_idx)
    test_subset = Subset(test_ds, test_idx)

    # Split train into train/val
    val_size = int(len(train_subset) * val_fraction)
    train_size = len(train_subset) - val_size
    train_split, val_split = random_split(
        train_subset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(
        train_split, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(
        val_split, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(
        test_subset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader


# -----------------------------------------------------------
# Evaluation
# -----------------------------------------------------------
def evaluate(model, loader, device) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


# -----------------------------------------------------------
# Training loop
# -----------------------------------------------------------
def train(
    epochs: int = EPOCHS,
    lr: float = LEARNING_RATE,
    batch_size: int = BATCH_SIZE,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print(f"Epochs: {epochs} | Batch: {batch_size} | LR: {lr} | Seed: {SEED}")

    set_seeds(SEED)

    train_loader, val_loader, test_loader = get_loaders(batch_size)
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches:   {len(val_loader)}")
    print(f"Test batches:  {len(test_loader)}")

    model = SmallCNN().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    curve = []          # Fix 4: per-epoch metrics
    start = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()

        avg_loss = running_loss / len(train_loader)
        val_acc = evaluate(model, val_loader, device)   # Fix 2: val, not test
        elapsed = time.time() - start

        curve.append({
            "epoch": epoch + 1,
            "loss": round(avg_loss, 4),
            "val_accuracy": round(val_acc, 4),
            "elapsed_seconds": round(elapsed, 1),
        })

        print(f"epoch {epoch+1:02d}/{epochs} | "
              f"loss={avg_loss:.4f} | "
              f"val_acc={val_acc:.4f} | "
              f"t={elapsed:.0f}s")

        # Fix 2: select checkpoint by validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), OUT_PATH)
            print(f"  -> saved best (val_acc={val_acc:.4f})")

    # Fix 2: test set touched ONCE, at the end
    model.load_state_dict(torch.load(OUT_PATH, map_location=device, weights_only=True))
    final_test_acc = evaluate(model, test_loader, device)

    print(f"\n{'='*60}")
    print(f"Best validation accuracy: {best_val_acc:.4f}")
    print(f"Final test accuracy:      {final_test_acc:.4f}")
    print(f"{'='*60}")

    # Fix 3: config.json
    config = {
        "classes": CLASSES,
        "num_classes": len(CLASSES),
        "input_shape": [1, 3, 32, 32],
        "normalization": {
            "mean": list(CIFAR_MEAN),
            "std": list(CIFAR_STD),
        },
        "training": {
            "epochs": epochs,
            "batch_size": batch_size,
            "optimizer": "Adam",
            "learning_rate": lr,
            "scheduler": "CosineAnnealingLR",
            "loss": "CrossEntropyLoss",
            "seed": SEED,
            "val_fraction": VAL_FRACTION,
            "dataset": "CIFAR-10 (classes 0-3)",
            "trained_on": datetime.utcnow().isoformat() + "Z",
        },
        "metrics": {
            "best_val_accuracy": round(best_val_acc, 4),
            "final_test_accuracy": round(final_test_acc, 4),
        },
        "parameters": sum(p.numel() for p in model.parameters()),
    }
    with open(CONFIG_PATH, 'w') as f:
        json.dump(config, f, indent=2)

    # Fix 4: curve.json
    with open(CURVE_PATH, 'w') as f:
        json.dump(curve, f, indent=2)

    print(f"Saved: {OUT_PATH}")
    print(f"Saved: {CONFIG_PATH}")
    print(f"Saved: {CURVE_PATH}")

    return model, config, curve


if __name__ == "__main__":
    train()

Overwriting /content/train.py


In [7]:
# ============================================================
# Run training
# ============================================================
# Expected: 4-6 minutes on T4. Final accuracy 78-82%.
# If test accuracy is below 75%, do not proceed — debug first.
# ============================================================

%cd /content
!python train.py

/content
Device: cpu
Epochs: 20 | Batch: 128 | LR: 0.001 | Seed: 42
100% 170M/170M [42:41<00:00, 66.6kB/s]
Train batches: 141
Val batches:   16
Test batches:  32
epoch 01/20 | loss=0.8522 | val_acc=0.7645 | t=42s
  -> saved best (val_acc=0.7645)
epoch 02/20 | loss=0.6170 | val_acc=0.7995 | t=83s
  -> saved best (val_acc=0.7995)
epoch 03/20 | loss=0.5207 | val_acc=0.8180 | t=125s
  -> saved best (val_acc=0.8180)
epoch 04/20 | loss=0.4710 | val_acc=0.8345 | t=166s
  -> saved best (val_acc=0.8345)
epoch 05/20 | loss=0.4015 | val_acc=0.8360 | t=208s
  -> saved best (val_acc=0.8360)
epoch 06/20 | loss=0.3648 | val_acc=0.8590 | t=251s
  -> saved best (val_acc=0.8590)
epoch 07/20 | loss=0.3137 | val_acc=0.8495 | t=293s
epoch 08/20 | loss=0.2666 | val_acc=0.8630 | t=336s
  -> saved best (val_acc=0.8630)
epoch 09/20 | loss=0.2275 | val_acc=0.8655 | t=378s
  -> saved best (val_acc=0.8655)
epoch 10/20 | loss=0.1953 | val_acc=0.8645 | t=420s
epoch 11/20 | loss=0.1617 | val_acc=0.8645 | t=462s
epoc

In [8]:
# ============================================================
#  Verify model loads and predicts correctly
# ============================================================
# Why: never trust a model you have not loaded back. If this
# fails, the local serving pipeline will also fail.
# ============================================================

import torch
import sys
import json
sys.path.insert(0, '/content')
from net import SmallCNN, CLASSES

model = SmallCNN()
state = torch.load('/content/baseline.pt', map_location='cpu', weights_only=True)
model.load_state_dict(state)
model.eval()

print("Model loaded")
print("Parameters:", sum(p.numel() for p in model.parameters()))

# Read config to confirm constants match
with open('/content/config.json') as f:
    config = json.load(f)
print("Config classes:", config["classes"])
print("Config normalization:", config["normalization"])
print("Final test accuracy:", config["metrics"]["final_test_accuracy"])

# Quick prediction check on one test image
import torchvision
from torchvision import transforms

ds = torchvision.datasets.CIFAR10(
    root='/content/data', train=False, download=False,
    transform=transforms.ToTensor())
img, true_label = ds[0]
tfm = transforms.Compose([transforms.Normalize((0.5,)*3, (0.5,)*3)])
x = tfm(img).unsqueeze(0)

with torch.no_grad():
    probs = torch.softmax(model(x), dim=1).squeeze(0)

print(f"True:      {CLASSES[true_label]}")
print(f"Predicted: {CLASSES[int(probs.argmax())]}")
print(f"Confidence: {float(probs.max()):.4f}")

Model loaded
Parameters: 618820
Config classes: ['airplane', 'automobile', 'bird', 'cat']
Config normalization: {'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]}
Final test accuracy: 0.8542
True:      cat
Predicted: cat
Confidence: 0.9876


In [9]:
# ============================================================
# Generate 20 sample images for local attack scripts
# ============================================================
# Why: attack scripts need real CIFAR-10 images to perturb.
# 5 per class = 20 total, enough for every attack demonstration.
# ============================================================

import torchvision
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import os

SAMPLES_DIR = '/content/samples'
os.makedirs(SAMPLES_DIR, exist_ok=True)

classes = ['airplane', 'automobile', 'bird', 'cat']
ds = torchvision.datasets.CIFAR10(
    root='/content/data', train=False, download=True,
    transform=transforms.ToTensor())

saved = {0: 0, 1: 0, 2: 0, 3: 0}
for img, label in ds:
    if label in saved and saved[label] < 5:
        np_img = (img.numpy().transpose(1, 2, 0) * 255).astype('uint8')
        Image.fromarray(np_img).save(
            f'{SAMPLES_DIR}/{classes[label]}_{saved[label]}.png')
        saved[label] += 1
    if all(v >= 5 for v in saved.values()):
        break

print(f'Saved to {SAMPLES_DIR}')
print(sorted(os.listdir(SAMPLES_DIR)))

Saved to /content/samples
['airplane_0.png', 'airplane_1.png', 'airplane_2.png', 'airplane_3.png', 'airplane_4.png', 'automobile_0.png', 'automobile_1.png', 'automobile_2.png', 'automobile_3.png', 'automobile_4.png', 'bird_0.png', 'bird_1.png', 'bird_2.png', 'bird_3.png', 'bird_4.png', 'cat_0.png', 'cat_1.png', 'cat_2.png', 'cat_3.png', 'cat_4.png']


In [10]:
# ============================================================
#  Zip samples for one-click download
# ============================================================

import shutil
shutil.make_archive('/content/samples', 'zip', '/content/samples')
print('Created /content/samples.zip')

Created /content/samples.zip


In [11]:
# ============================================================
# Download artifacts to your local machine
# ============================================================
# Four files: baseline.pt, config.json, curve.json, samples.zip
# If a download does not start, use the Files panel on the left.
# ============================================================

from google.colab import files

files.download('/content/baseline.pt')
files.download('/content/config.json')
files.download('/content/curve.json')
files.download('/content/samples.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>